In [5]:
# simplified_medical_train.py
"""
Medical-focused model training with KNN instead of SVM
Trains ONLY required models: 1 Baseline, 2 Classical (Random Forest + KNN), 1 Ensemble
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix
import joblib
import json
import warnings
import os

warnings.filterwarnings('ignore')

class MedicalModelTrainer:
    def __init__(self):
        # ALL diseases from BRFSS
        self.all_diseases = [
            'Diabetes', 'HeartDisease', 'Stroke', 'Asthma',
            'HeartAttack', 'COPD', 'Depression', 'KidneyDisease',
            'Arthritis', 'SkinCancer'
        ]
        
        # Model configurations
        self.models_config = {
            'Baseline': {
                'name': 'Logistic Regression',
                'class': 'LogisticRegression',
                'params': {'max_iter': 1000, 'class_weight': 'balanced'}
            },
            'Classical_1': {
                'name': 'Random Forest',
                'class': 'RandomForestClassifier',
                'params': {'n_estimators': 200, 'class_weight': 'balanced', 'max_depth': 15}
            },
            'Classical_2': {
                'name': 'KNN',
                'class': 'KNeighborsClassifier',
                'params': {'n_neighbors': 15, 'weights': 'distance'}  # KNN instead of SVM
            },
            'Ensemble': {
                'name': 'Stacking Classifier',
                'class': 'StackingClassifier',
                'params': {}
            }
        }
    
    def load_all_features_data(self):
        """Load the complete dataset with ALL features"""
        print("Loading complete feature dataset...")
        
        # Load individual BRFSS data
        individual_data = pd.read_csv('../data/processed/brfss_individual_cleaned.csv')
        
        # Load EPA data
        epa_data = pd.read_csv('../data/raw/annual_aqi_by_county_2023.csv')
        
        # Merge datasets
        merged_data = self._merge_datasets(individual_data, epa_data)
        
        print(f"Complete dataset loaded: {merged_data.shape[0]:,} samples, {merged_data.shape[1]} features")
        
        return merged_data
    
    def _merge_datasets(self, brfss_data, epa_data):
        """Merge BRFSS with EPA data"""
        # Process EPA data
        epa_processed = epa_data.groupby('State').agg({
            'Max AQI': 'mean',
            'Median AQI': 'mean',
            '90th Percentile AQI': 'mean',
            'Good Days': 'mean',
            'Unhealthy Days': 'mean'
        }).reset_index()
        
        # Merge
        merged = brfss_data.merge(epa_processed, on='State', how='left')
        
        # Fill missing values
        for col in merged.columns:
            if merged[col].isnull().any():
                if merged[col].dtype in ['float64', 'int64']:
                    merged[col].fillna(merged[col].median(), inplace=True)
                else:
                    merged[col].fillna(merged[col].mode()[0], inplace=True)
        
        return merged
    
    def prepare_features_and_targets(self, data):
        """Prepare ALL features and disease targets"""
        print("\nPreparing features and targets...")
        
        # Define targets (disease columns)
        target_columns = [col for col in data.columns if col in self.all_diseases]
        
        # Define ALL features (exclude targets and identifiers)
        exclude_columns = target_columns + [
            'FIPS', 'State', 'State_FIPS', 'County_FIPS',
            'ID', 'SEQNO', 'RecordID'
        ]
        
        feature_columns = [col for col in data.columns if col not in exclude_columns]
        
        # Handle categorical features
        categorical_features = []
        numeric_features = []
        
        for col in feature_columns:
            if data[col].dtype == 'object':
                categorical_features.append(col)
            else:
                numeric_features.append(col)
        
        print(f"Total features: {len(feature_columns)}")
        print(f"  Numeric: {len(numeric_features)}")
        print(f"  Categorical: {len(categorical_features)}")
        print(f"Target diseases: {len(target_columns)}")
        
        return feature_columns, target_columns, categorical_features, numeric_features
    
    def preprocess_data(self, X, categorical_features, numeric_features):
        """Preprocess ALL features"""
        print("\nPreprocessing data...")
        
        # Preprocessing pipeline
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, numeric_features),
                ('cat', categorical_transformer, categorical_features)
            ])
        
        # Fit and transform
        X_processed = preprocessor.fit_transform(X)
        
        # Get feature names after preprocessing
        if hasattr(preprocessor.named_transformers_['cat']['onehot'], 'get_feature_names_out'):
            cat_features_out = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
        else:
            cat_features_out = []
        
        all_feature_names = list(numeric_features) + list(cat_features_out)
        
        print(f"Processed features: {X_processed.shape[1]}")
        
        return X_processed, preprocessor, all_feature_names
    
    def train_models_medical_focus(self):
        """Train models with medical focus (RECALL prioritized)"""
        print("\n" + "="*80)
        print("MEDICAL-FOCUSED MODEL TRAINING")
        print("="*80)
        
        # Load data
        data = self.load_all_features_data()
        
        # Prepare features and targets
        feature_columns, target_columns, cat_features, num_features = self.prepare_features_and_targets(data)
        
        X = data[feature_columns]
        
        # Preprocess
        X_processed, preprocessor, feature_names = self.preprocess_data(X, cat_features, num_features)
        
        results = {}
        
        for disease in target_columns:
            print(f"\n{'='*60}")
            print(f"DISEASE: {disease}")
            print(f"{'='*60}")
            
            y = data[disease].fillna(0)
            
            # Check class balance and skip if only one class
            unique_classes = y.unique()
            if len(unique_classes) < 2:
                print(f"⚠️  Skipping {disease}: Only {len(unique_classes)} class(es) in data. "
                      f"Classes: {unique_classes}")
                print(f"   This disease might be too rare in your dataset.")
                print(f"   Consider: 1) Using more data, 2) Using a different dataset, "
                      f"3) Focusing on more common diseases")
                continue
            
            # Check class balance
            disease_prevalence = y.mean() * 100
            print(f"Prevalence: {disease_prevalence:.1f}%")
            print(f"Class distribution: {y.value_counts().to_dict()}")
            
            # Check if we have enough positive cases
            positive_cases = y.sum()
            if positive_cases < 10:
                print(f"⚠️  Warning: Only {positive_cases} positive cases for {disease}")
                print(f"   Models may not generalize well")
            
            # Split data
            try:
                # Use stratification only if we have enough positive cases
                use_stratify = (disease_prevalence > 5) and (positive_cases >= 20)
                
                X_train, X_test, y_train, y_test = train_test_split(
                    X_processed, y, test_size=0.2, random_state=42,
                    stratify=y if use_stratify else None
                )
                
                # Double check that both train and test have at least 2 classes
                if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
                    print(f"⚠️  Skipping {disease}: Train or test set has only one class after split")
                    print(f"   Train classes: {np.unique(y_train)}, Test classes: {np.unique(y_test)}")
                    continue
                
            except ValueError as e:
                print(f"⚠️  Error splitting data for {disease}: {e}")
                print(f"   Using random split without stratification")
                X_train, X_test, y_train, y_test = train_test_split(
                    X_processed, y, test_size=0.2, random_state=42
                )
                
                # Check again after split
                if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
                    print(f"⚠️  Skipping {disease}: Train or test set has only one class")
                    continue
            
            print(f"Train: {X_train.shape[0]:,} samples (Positive: {y_train.sum()})")
            print(f"Test: {X_test.shape[0]:,} samples (Positive: {y_test.sum()})")
            
            # Train models
            try:
                disease_results = self._train_disease_models(X_train, X_test, y_train, y_test, disease)
                results[disease] = disease_results
                
                # Save models
                self._save_disease_models(disease, disease_results, preprocessor, feature_names)
            except Exception as e:
                print(f"❌ Failed to train models for {disease}: {e}")
                print(f"   Error type: {type(e).__name__}")
                print(f"   Skipping this disease...")
        
        # Generate comprehensive evaluation
        if results:
            self._generate_medical_evaluation(results)
        else:
            print("\n❌ No diseases were successfully trained!")
            print("   Check your data for class imbalance issues.")
        
        return results
    
    def _train_disease_models(self, X_train, X_test, y_train, y_test, disease_name):
        """Train all required models for a disease"""
        from sklearn.linear_model import LogisticRegression
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.neighbors import KNeighborsClassifier
        from sklearn.ensemble import StackingClassifier
        from sklearn.dummy import DummyClassifier
        
        results = {}
        
        # Create a simple baseline model for comparison
        print("\n[0/4] Creating simple baseline (Dummy Classifier)...")
        dummy = DummyClassifier(strategy='stratified', random_state=42)
        dummy.fit(X_train, y_train)
        dummy_metrics = self._evaluate_medical_model(dummy, X_test, y_test, 'Dummy Baseline')
        results['Dummy'] = {'model': dummy, 'metrics': dummy_metrics}
        
        # 1. BASELINE: Logistic Regression
        print("\n[1/4] Training Baseline (Logistic Regression)...")
        try:
            baseline = LogisticRegression(**self.models_config['Baseline']['params'])
            baseline.fit(X_train, y_train)
            baseline_metrics = self._evaluate_medical_model(baseline, X_test, y_test, 'Logistic Regression')
            results['Baseline'] = {'model': baseline, 'metrics': baseline_metrics}
        except Exception as e:
            print(f"   ❌ Logistic Regression failed: {e}")
            print(f"   Using Dummy classifier as baseline instead")
            results['Baseline'] = results['Dummy']
        
        # 2. CLASSICAL 1: Random Forest
        print("\n[2/4] Training Classical Model 1 (Random Forest)...")
        try:
            rf = RandomForestClassifier(**self.models_config['Classical_1']['params'])
            rf.fit(X_train, y_train)
            rf_metrics = self._evaluate_medical_model(rf, X_test, y_test, 'Random Forest')
            results['RandomForest'] = {'model': rf, 'metrics': rf_metrics}
        except Exception as e:
            print(f"   ❌ Random Forest failed: {e}")
            print(f"   Skipping Random Forest for this disease")
        
        # 3. CLASSICAL 2: KNN (Instead of SVM)
        print("\n[3/4] Training Classical Model 2 (KNN)...")
        try:
            # Scale data for KNN
            from sklearn.preprocessing import StandardScaler
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            knn = KNeighborsClassifier(**self.models_config['Classical_2']['params'])
            knn.fit(X_train_scaled, y_train)
            knn_metrics = self._evaluate_medical_model(knn, X_test_scaled, y_test, 'KNN')
            results['KNN'] = {'model': knn, 'metrics': knn_metrics, 'scaler': scaler}
        except Exception as e:
            print(f"   ❌ KNN failed: {e}")
            print(f"   Skipping KNN for this disease")
        
        # 4. ENSEMBLE: Stacking Classifier (only if we have at least 2 models)
        print("\n[4/4] Training Ensemble Model (Stacking)...")
        try:
            # Collect available models for stacking
            available_models = []
            if 'RandomForest' in results:
                available_models.append(('rf', results['RandomForest']['model']))
            if 'KNN' in results:
                # For KNN, we need to use the scaled data
                if 'scaler' in results['KNN']:
                    # We'll need to handle KNN differently since it uses scaled data
                    print(f"   Note: KNN uses scaled data, adjusting for stacking...")
                    # Create a pipeline for KNN in stacking
                    knn_pipeline = Pipeline([
                        ('scaler', results['KNN']['scaler']),
                        ('knn', results['KNN']['model'])
                    ])
                    available_models.append(('knn', knn_pipeline))
            
            # Only create stacking if we have at least 2 base models
            if len(available_models) >= 2:
                ensemble = StackingClassifier(
                    estimators=available_models,
                    final_estimator=LogisticRegression(max_iter=1000),
                    cv=3
                )
                ensemble.fit(X_train, y_train)
                ensemble_metrics = self._evaluate_medical_model(ensemble, X_test, y_test, 'Ensemble')
                results['Ensemble'] = {'model': ensemble, 'metrics': ensemble_metrics}
            else:
                print(f"   ⚠️  Not enough base models for stacking (need at least 2, have {len(available_models)})")
                print(f"   Skipping Ensemble model")
        except Exception as e:
            print(f"   ❌ Ensemble failed: {e}")
            print(f"   Skipping Ensemble for this disease")
        
        # Print comparison
        if len(results) > 1:  # More than just the dummy classifier
            self._print_model_comparison(results, disease_name)
        else:
            print(f"\n⚠️  No successful models for {disease_name}")
            print(f"   All models failed. Check data quality and class balance.")
        
        return results
    
    def _evaluate_medical_model(self, model, X_test, y_test, model_name):
        """Evaluate model with medical focus on RECALL"""
        from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix
        
        try:
            y_pred = model.predict(X_test)
            y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
            
            # Calculate metrics - convert to Python native types immediately
            recall = float(recall_score(y_test, y_pred, zero_division=0))
            precision = float(precision_score(y_test, y_pred, zero_division=0))
            f1 = float(f1_score(y_test, y_pred, zero_division=0))
            accuracy = float(np.mean(y_pred == y_test))
            
            metrics = {
                'recall': recall,
                'precision': precision,
                'f1': f1,
                'accuracy': accuracy
            }
            
            # ROC-AUC if probabilities available
            if y_pred_proba is not None:
                try:
                    metrics['auc'] = float(roc_auc_score(y_test, y_pred_proba))
                except:
                    metrics['auc'] = 0.5  # Default for binary classification
            
            # Confusion matrix - convert to Python int
            tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
            metrics['confusion_matrix'] = {
                'tn': int(tn), 
                'fp': int(fp), 
                'fn': int(fn), 
                'tp': int(tp)
            }
            
            # Medical-specific metrics
            metrics['sensitivity'] = recall  # Same as recall
            specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0
            metrics['specificity'] = specificity
            
            # Medical score (weighted: 70% recall, 30% precision)
            metrics['medical_score'] = float((0.7 * recall) + (0.3 * precision))
            
            # False Negative Rate (critical for medical)
            fnr = float(fn / (fn + tp)) if (fn + tp) > 0 else 0.0
            metrics['false_negative_rate'] = fnr
            
            print(f"  {model_name}:")
            print(f"    Recall: {recall:.3f} | Precision: {precision:.3f} | F1: {f1:.3f}")
            print(f"    Medical Score: {metrics['medical_score']:.3f} | FNR: {fnr:.3f}")
            
            return metrics
        except Exception as e:
            print(f"  ❌ Evaluation failed for {model_name}: {e}")
            # Return default metrics
            return {
                'recall': 0.0,
                'precision': 0.0,
                'f1': 0.0,
                'accuracy': 0.0,
                'auc': 0.5,
                'confusion_matrix': {'tn': 0, 'fp': 0, 'fn': 0, 'tp': 0},
                'sensitivity': 0.0,
                'specificity': 0.0,
                'medical_score': 0.0,
                'false_negative_rate': 1.0
            }
    
    def _convert_to_python_types(self, obj):
        """Convert NumPy/pandas types to Python native types for JSON serialization"""
        if isinstance(obj, dict):
            return {key: self._convert_to_python_types(value) for key, value in obj.items()}
        elif isinstance(obj, list):
            return [self._convert_to_python_types(item) for item in obj]
        elif isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif pd.isna(obj):
            return None
        else:
            return obj
    
    def _print_model_comparison(self, results, disease_name):
        """Print detailed model comparison"""
        # Filter out dummy classifier for comparison
        models_to_compare = {k: v for k, v in results.items() if k != 'Dummy'}
        
        if not models_to_compare:
            print(f"\n⚠️  No valid models to compare for {disease_name}")
            return
            
        print(f"\n📊 {disease_name} - Model Comparison (Medical Focus):")
        print("-" * 100)
        print(f"{'Model':<20} {'Recall':<10} {'Precision':<10} {'F1':<10} {'Medical Score':<15} {'FNR':<10}")
        print("-" * 100)
        
        for model_name, info in models_to_compare.items():
            metrics = info['metrics']
            print(f"{model_name:<20} {metrics['recall']:<10.3f} {metrics['precision']:<10.3f} "
                  f"{metrics['f1']:<10.3f} {metrics['medical_score']:<15.3f} "
                  f"{metrics['false_negative_rate']:<10.3f}")
        
        # Find best model by recall (excluding dummy)
        if models_to_compare:
            best_recall = max(models_to_compare.items(), key=lambda x: x[1]['metrics']['recall'])
            print(f"\n🎯 **Best for medical safety (highest recall):** {best_recall[0]} "
                  f"(Recall: {best_recall[1]['metrics']['recall']:.3f}, "
                  f"FNR: {best_recall[1]['metrics']['false_negative_rate']:.3f})")
            
            # Find best model by medical score
            best_medical = max(models_to_compare.items(), key=lambda x: x[1]['metrics']['medical_score'])
            print(f"⚖️  **Best balanced (medical score):** {best_medical[0]} "
                  f"(Score: {best_medical[1]['metrics']['medical_score']:.3f})")
            
            # Medical recommendation
            if best_recall[1]['metrics']['recall'] < 0.8:
                print("⚠️  **WARNING:** All models have recall < 80%. Consider additional clinical screening.")
    
    def _save_disease_models(self, disease, results, preprocessor, feature_names):
        """Save trained models and preprocessing info"""
        save_dir = f"models/medical_focus/{disease}"
        os.makedirs(save_dir, exist_ok=True)
        
        # Save each model
        for model_name, info in results.items():
            model_path = f"{save_dir}/{model_name}.pkl"
            joblib.dump(info['model'], model_path)
        
        # Save preprocessor and feature names
        joblib.dump(preprocessor, f"{save_dir}/preprocessor.pkl")
        joblib.dump(feature_names, f"{save_dir}/feature_names.pkl")
        
        # Save metrics - convert to Python types first
        metrics = {}
        for name, info in results.items():
            # Convert all metrics to Python native types
            metrics[name] = self._convert_to_python_types(info['metrics'])
        
        with open(f"{save_dir}/metrics.json", 'w') as f:
            json.dump(metrics, f, indent=2, default=str)
        
        print(f"  Models saved to: {save_dir}/")
    
    def _generate_medical_evaluation(self, results):
        """Generate comprehensive medical evaluation report"""
        print("\n" + "="*80)
        print("COMPREHENSIVE MEDICAL EVALUATION REPORT")
        print("="*80)
        
        if not results:
            print("❌ No results to generate evaluation report!")
            return
            
        report = {
            'summary': {},
            'disease_analysis': {},
            'recommendations': []
        }
        
        for disease, disease_results in results.items():
            # Filter out dummy classifier
            valid_results = {k: v for k, v in disease_results.items() if k != 'Dummy'}
            
            if not valid_results:
                print(f"\n⚠️  No valid models for {disease}, skipping from report")
                continue
                
            # Collect best metrics
            best_recall = max(valid_results.items(), 
                            key=lambda x: x[1]['metrics']['recall'])
            best_medical = max(valid_results.items(),
                             key=lambda x: x[1]['metrics']['medical_score'])
            
            report['disease_analysis'][disease] = {
                'best_recall_model': best_recall[0],
                'best_recall_value': float(best_recall[1]['metrics']['recall']),
                'best_medical_model': best_medical[0],
                'best_medical_score': float(best_medical[1]['metrics']['medical_score']),
                'lowest_fnr': min(valid_results.items(),
                                key=lambda x: x[1]['metrics']['false_negative_rate'])[0]
            }
            
            # Medical classification
            recall = best_recall[1]['metrics']['recall']
            if recall >= 0.9:
                status = "Excellent"
            elif recall >= 0.8:
                status = "Good"
            elif recall >= 0.7:
                status = "Moderate"
            else:
                status = "Poor"
            
            print(f"\n{disease}:")
            print(f"  Best Recall: {best_recall[0]} ({recall:.3f}) - {status}")
            print(f"  Best Medical Score: {best_medical[0]} ({best_medical[1]['metrics']['medical_score']:.3f})")
        
        # Save report - convert to Python types
        report_dir = "reports/medical_evaluation"
        os.makedirs(report_dir, exist_ok=True)
        
        # Convert report to Python native types
        report_converted = self._convert_to_python_types(report)
        
        with open(f"{report_dir}/comprehensive_report.json", 'w') as f:
            json.dump(report_converted, f, indent=2, default=str)
        
        print(f"\n📄 Report saved to: {report_dir}/")

def main():
    """Main training execution"""
    trainer = MedicalModelTrainer()
    results = trainer.train_models_medical_focus()
    
    print("\n" + "="*80)
    print("TRAINING COMPLETE")
    print("="*80)
    
    if results:
        print(f"\n✅ Successfully trained models for {len(results)} diseases:")
        for disease in results.keys():
            print(f"  ✓ {disease}")
        
        print("\nModels trained for medical application:")
        print("✓ 1 Baseline Model: Logistic Regression (or Dummy if fails)")
        print("✓ 2 Classical Models: Random Forest + KNN (not SVM)")
        print("✓ 1 Ensemble Model: Stacking Classifier (if enough base models)")
        print("✓ Medical focus: RECALL prioritized (70% weight)")
        print("✓ All features used for training")
        print("\nModel directory: models/medical_focus/")
        print("Evaluation reports: reports/medical_evaluation/")
    else:
        print("\n❌ No diseases were successfully trained!")
        print("\nCommon issues and solutions:")
        print("1. **Class imbalance**: Some diseases may have too few positive cases")
        print("   Solution: Check your data, consider oversampling or different dataset")
        print("2. **Data quality**: Missing values or incorrect encoding")
        print("   Solution: Run data_merge.py to ensure proper preprocessing")
        print("3. **Skin Cancer specific**: BRFSS might have very few skin cancer cases")
        print("   Solution: Focus on more common diseases like Diabetes, HeartDisease")

if __name__ == "__main__":
    main()


MEDICAL-FOCUSED MODEL TRAINING
Loading complete feature dataset...
Complete dataset loaded: 433,323 samples, 75 features

Preparing features and targets...
Total features: 61
  Numeric: 59
  Categorical: 2
Target diseases: 10

Preprocessing data...
Processed features: 63

DISEASE: HeartDisease
Prevalence: 5.4%
Class distribution: {0.0: 409869, 1.0: 23454}
Train: 346,658 samples (Positive: 18763.0)
Test: 86,665 samples (Positive: 4691.0)

[0/4] Creating simple baseline (Dummy Classifier)...
  Dummy Baseline:
    Recall: 0.055 | Precision: 0.057 | F1: 0.056
    Medical Score: 0.056 | FNR: 0.945

[1/4] Training Baseline (Logistic Regression)...
  Logistic Regression:
    Recall: 0.870 | Precision: 0.252 | F1: 0.391
    Medical Score: 0.685 | FNR: 0.130

[2/4] Training Classical Model 1 (Random Forest)...
  Random Forest:
    Recall: 0.799 | Precision: 0.284 | F1: 0.420
    Medical Score: 0.645 | FNR: 0.201

[3/4] Training Classical Model 2 (KNN)...
  KNN:
    Recall: 0.081 | Precision: 0